In [3]:
import java.io.File
import java.time.LocalDate
import java.time.format.TextStyle
import java.util.Locale

val docsDir = listOf(File("docs"), File("/Users/admin/Documents/weekly/docs")).first { it.isDirectory }
val rootDir = docsDir.parentFile
val issueRegex = Regex("""issue-(\d+)\.md""")

// 一次 git log 取出每个文件首次加入仓库的日期（日志从新到旧，后出现的覆盖，即保留最早一次）
val addedDates = mutableMapOf<String, String>()
val gitOutput = ProcessBuilder(
    "git", "log", "--diff-filter=A", "--name-only", "--format=D %ad", "--date=short", "--", "docs/issue-*.md"
).directory(rootDir).redirectErrorStream(true).start()
    .inputStream.bufferedReader().readLines()

var currentDate = ""
for (line in gitOutput) {
    when {
        line.startsWith("D ") -> currentDate = line.removePrefix("D ").trim()
        line.isNotBlank() -> addedDates[File(line.trim()).name] = currentDate
    }
}

// 标题取自 README.md 目录，形如：- 第 417 期：[不要让 AI 代表你](docs/issue-417.md)
val titleRegex = Regex("""^- 第 (\d+) 期：\[(.+)]\(docs/issue-\d+\.md\)\s*$""")
val titles: Map<Int, String> = File(rootDir, "README.md").readLines()
    .mapNotNull { titleRegex.matchEntire(it) }
    .associate { it.groupValues[1].toInt() to it.groupValues[2] }

fun publishInfo(fileName: String): String {
    val date = addedDates[fileName] ?: return "发布时间 未知"
    val weekday = LocalDate.parse(date).dayOfWeek.getDisplayName(TextStyle.FULL, Locale.CHINA)
    return "发布时间 $date $weekday"
}

val issues = docsDir.listFiles()
    .orEmpty()
    .mapNotNull { f -> issueRegex.matchEntire(f.name)?.let { it.groupValues[1].toInt() to f.name } }
    .sortedByDescending { it.first }

println("共 ${issues.size} 期")
issues.forEach { (num, name) ->
    println("第 $num 期  ${publishInfo(name)}  ${titles[num] ?: "（无标题）"}")
}

共 417 期
第 417 期  发布时间 2026-10-30 星期五  不要让 AI 代表你
第 416 期  发布时间 2026-10-23 星期五  编程已经面目全非
第 415 期  发布时间 2026-10-16 星期五  3D 版《看不见的城市》
第 414 期  发布时间 2026-10-08 星期四  Jev 决策模型有什么用
第 413 期  发布时间 2026-09-18 星期五  再见了，React Native
第 412 期  发布时间 2026-09-11 星期五  禁止 issue，只用 PR
第 411 期  发布时间 2026-09-04 星期五  OpenClaw 2.0 是一个缩影
第 410 期  发布时间 2026-08-28 星期五  你需要知道的 AI 三种机制
第 409 期  发布时间 2026-08-21 星期五  程序员的职业未来
第 408 期  发布时间 2026-08-14 星期五  你需要知道的 AI 缓存知识
第 407 期  发布时间 2026-08-07 星期五  国家为什么需要开源软件？
第 406 期  发布时间 2026-07-31 星期五  道可，道非，常道
第 405 期  发布时间 2026-07-24 星期五  资源，社会公平与算力
第 404 期  发布时间 2026-07-17 星期五  你需要知道的 AI 内存知识
第 403 期  发布时间 2026-07-10 星期五  为什么 Dropbox 不成功
第 402 期  发布时间 2026-07-03 星期五  我在智念 AI 的日子（小说）
第 401 期  发布时间 2026-06-26 星期五  如何赚到10亿美元
第 400 期  发布时间 2026-06-12 星期五  rsync 的争论
第 399 期  发布时间 2026-06-05 星期五  中国 AI 大厂访问记
第 398 期  发布时间 2026-05-29 星期五  Token 费用难以负担
第 397 期  发布时间 2026-05-22 星期五  财富正在向 AI 集中
第 396 期  发布时间 2026-05-15 星期五  互联网通信的替代方案
第 395 期  发布时间 2026-05-08 星期五  软件开发的第三种方式
第 394 期

In [7]:
// 依赖上一个单元格里的 docsDir、issues、titles、publishInfo，请先运行上一个单元格
val searchKeyword = "arch linux"   // 修改这里：要搜索的关键字（不区分大小写）
val contextLen = 30          // 命中处前后各取多少个字符作为上下文
val maxSnippets = 3          // 每期最多展示几条摘要

fun escapeHtml(s: String) = s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

fun highlight(text: String, regex: Regex): String {
    val sb = StringBuilder()
    var last = 0
    for (m in regex.findAll(text)) {
        sb.append(escapeHtml(text.substring(last, m.range.first)))
        sb.append("<mark>").append(escapeHtml(m.value)).append("</mark>")
        last = m.range.last + 1
    }
    sb.append(escapeHtml(text.substring(last)))
    return sb.toString()
}

// 去掉图片和链接地址，只保留链接文字，并把换行、连续空白压成一个空格，避免命中 URL
fun cleanText(raw: String): String = raw
    .replace(Regex("""!\[[^\]]*]\([^)]*\)"""), "")
    .replace(Regex("""\[([^\]]*)]\([^)]*\)"""), "\$1")
    .replace(Regex("""\s+"""), " ")

val issueTexts: Map<Int, String> = issues.associate { (num, name) -> num to cleanText(File(docsDir, name).readText()) }

fun searchIssues(keyword: String): String {
    require(keyword.isNotBlank()) { "关键字不能为空" }
    val regex = Regex(Regex.escape(keyword.trim()), RegexOption.IGNORE_CASE)
    val body = StringBuilder()
    var hitIssues = 0
    var hitTotal = 0

    for ((num, name) in issues) {
        val text = issueTexts.getValue(num)
        val matches = regex.findAll(text).toList()
        val title = titles[num] ?: "（无标题）"
        val titleHit = regex.containsMatchIn(title)
        if (matches.isEmpty() && !titleHit) continue
        hitIssues++
        hitTotal += matches.size

        val snippets = mutableListOf<String>()
        var lastEnd = -1
        for (m in matches) {
            if (m.range.first <= lastEnd) continue
            if (snippets.size >= maxSnippets) break
            val start = maxOf(0, m.range.first - contextLen)
            val end = minOf(text.length, m.range.last + 1 + contextLen)
            snippets += (if (start > 0) "…" else "") +
                highlight(text.substring(start, end), regex) +
                (if (end < text.length) "…" else "")
            lastEnd = end - 1
        }

        body.append("<div style=\"margin:10px 0\">")
        body.append("<b>第 $num 期</b> ")
        body.append("<span style=\"opacity:.6\">${escapeHtml(publishInfo(name))}</span> ")
        body.append("<b>${highlight(title, regex)}</b> ")
        body.append("<span style=\"opacity:.6\">（命中 ${matches.size} 处）</span>")
        if (snippets.isNotEmpty()) {
            body.append("<ul style=\"margin:4px 0\">")
            snippets.forEach { body.append("<li>$it</li>") }
            if (matches.size > snippets.size) body.append("<li style=\"opacity:.6\">……</li>")
            body.append("</ul>")
        }
        body.append("</div>")
    }

    val header = if (hitIssues == 0) "未找到包含「${escapeHtml(keyword)}」的周刊"
    else "关键字「<mark>${escapeHtml(keyword)}</mark>」：命中 $hitIssues 期，共 $hitTotal 处（按期数递减）"
    return "<h3>$header</h3>$body"
}

HTML(searchIssues(searchKeyword))

关键字「 arch linux 」：命中 6 期，共 9 处（按期数递减） 第 336 期 发布时间 2025-02-07 星期五 面对 AI，互联网正在衰落 （命中 2 处） …春联生成工具。（@itfoxnet 投稿） ## 资源 1、 Arch Linux 初学者配置指南 这是一组系列文章，向初学者介绍如何配置 A… …… 第 272 期 发布时间 2023-09-22 星期五 Unity 的安装费，游戏业的缩影 （命中 2 处） …个点对点网络，本文介绍它的技术架构，还算好懂。 6、如何在 Arch Linux 安装 Steam（英文） 本文教大家把世界最大游戏平台 S… …（英文） 本文教大家把世界最大游戏平台 Steam 安装到 Arch Linux 系统上。 7、《Rust 还是 Go？》（英文） 本文作者… 第 219 期 发布时间 2022-08-19 星期五 如何防止帐号被黑 （命中 2 处） …6个月内发明的。 -- Hacker News 读者 3、 Arch Linux 的优点和缺点都是它的滚动更新机制。一方面，你总是能让系统保… …某个软件保持特定版本，所以不适合用作生产环境服务器。 --《 Arch Linux 为何存在》 4、 如果你是一个工程师，那么在成为资深工程师… 第 121 期 发布时间 2020-08-21 星期五 为什么人类没有越来越闲？ （命中 1 处） …A，后端开发者现在也可以写前端了。 3、Manjaro 与 Arch Linux 的区别（英文） Manjaro 与 Arch 都是流行的 … 第 30 期 发布时间 2018-11-09 星期五 为什么谷歌做不好社交软件？ （命中 1 处） … Manjaro 是一个新的 Linux 发行版，内核采用 Arch Linux ，UI 采用 Deepin，集灵活性和易用的 UI 于一体。… 第 23 期 发布时间 2018-10-13 星期六 统计学上的人生最大决定因素 （命中 1 处） …Linux 系统，但是目前只支持 Ubuntu。 2、fse arch Linux 的桌面应用，用来文件搜索，可以替代 find 命令。 3、…